### Instruction

> 1. Rename assignment-01-XXX-YYY.ipynb where XXX is your student ID and YYY is your Chinese name.
> 2. The deadline of Assignment-01 is 23:59pm, 04-02-2026
> 3. In this assignment, you will
>    1) Explore Wikipedia text data
>    2) Build language models
>    3) Build NB and LR classifiers
>    4) Build embedding model for document classification
> 4. Submit your homework as html file just like our exercise did.
> Download the preprocessed data, enwiki-train.json and enwiki-test.json from the Assignment-01 folder. In the data file, each line contains a Wikipedia page with attributes, title, label, and text. There are 9000 records in the train file and 1000 records in test file with ten categories.

- Please print out all your outputs in the notebook and save it.

### Task1 - Data exploring

> 1) Print out how many documents are in each class  (for both train and test dataset)

In [31]:
from collections import Counter
import json
from pathlib import Path

DATA_DIR = Path.cwd()
TRAIN_PATH = DATA_DIR / "enwiki-train.json"
TEST_PATH = DATA_DIR / "enwiki-test.json"


def load_jsonl(file_path: Path):
    records = []
    with file_path.open("r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                records.append(json.loads(line))
    return records


def count_documents_by_label(records):
    return Counter(record["label"] for record in records)


train_records = load_jsonl(TRAIN_PATH)
test_records = load_jsonl(TEST_PATH)
train_label_counts = count_documents_by_label(train_records)
test_label_counts = count_documents_by_label(test_records)

print("Training dataset - number of documents in each class:")
for label, count in sorted(train_label_counts.items()):
    print(f"{label}: {count}")

print("\nTesting dataset - number of documents in each class:")
for label, count in sorted(test_label_counts.items()):
    print(f"{label}: {count}")


Training dataset - number of documents in each class:
Actor: 79
Animal: 82
Artist: 457
Book: 858
Disease: 202
Film: 2752
Food: 121
Politician: 3441
Software: 239
Writer: 769

Testing dataset - number of documents in each class:
Actor: 1
Animal: 11
Artist: 63
Book: 117
Disease: 18
Film: 296
Food: 16
Politician: 383
Software: 27
Writer: 68


> 2) Print out the average number of sentences in each class.
>    You may need to use sentence tokenization tools from spacy for both train and test dataset.



In [32]:
import spacy
from collections import defaultdict

nlp = spacy.blank("en")
if "sentencizer" not in nlp.pipe_names:
    nlp.add_pipe("sentencizer")


def average_sentence_count_by_label(records, nlp, batch_size: int = 64):
    sentence_totals = defaultdict(int)
    document_totals = defaultdict(int)

    for record, doc in zip(records, nlp.pipe((record["text"] for record in records), batch_size=batch_size)):
        label = record["label"]
        sentence_totals[label] += sum(1 for sent in doc.sents if sent.text.strip())
        document_totals[label] += 1

    return {
        label: sentence_totals[label] / document_totals[label]
        for label in sorted(document_totals)
    }


train_avg_sentence_counts = average_sentence_count_by_label(train_records, nlp)
test_avg_sentence_counts = average_sentence_count_by_label(test_records, nlp)

print("Training dataset - average number of sentences in each class:")
for label, avg_count in train_avg_sentence_counts.items():
    print(f"{label}: {avg_count:.2f}")

print("\nTesting dataset - average number of sentences in each class:")
for label, avg_count in test_avg_sentence_counts.items():
    print(f"{label}: {avg_count:.2f}")


Training dataset - average number of sentences in each class:
Actor: 69.33
Animal: 67.06
Artist: 185.94
Book: 205.08
Disease: 347.86
Film: 177.88
Food: 153.44
Politician: 222.86
Software: 201.13
Writer: 216.11

Testing dataset - average number of sentences in each class:
Actor: 177.00
Animal: 62.82
Artist: 174.97
Book: 197.96
Disease: 325.89
Film: 173.68
Food: 165.12
Politician: 233.03
Software: 204.00
Writer: 220.60


> 3) Print out the average number of tokens in each class for both train and test dataset.

In [33]:
def average_token_count_by_label(records, nlp, batch_size: int = 64):
    token_totals = defaultdict(int)
    document_totals = defaultdict(int)

    for record, doc in zip(records, nlp.pipe((record["text"] for record in records), batch_size=batch_size)):
        label = record["label"]
        token_totals[label] += sum(1 for token in doc if not token.is_space)
        document_totals[label] += 1

    return {
        label: token_totals[label] / document_totals[label]
        for label in sorted(document_totals)
    }


train_avg_token_counts = average_token_count_by_label(train_records, nlp)
test_avg_token_counts = average_token_count_by_label(test_records, nlp)

print("Training dataset - average number of tokens in each class:")
for label, avg_count in train_avg_token_counts.items():
    print(f"{label}: {avg_count:.2f}")

print("\nTesting dataset - average number of tokens in each class:")
for label, avg_count in test_avg_token_counts.items():
    print(f"{label}: {avg_count:.2f}")


Training dataset - average number of tokens in each class:
Actor: 1713.95
Animal: 1466.94
Artist: 4905.29
Book: 5381.72
Disease: 8204.41
Film: 4525.12
Food: 3533.75
Politician: 5769.80
Software: 4927.05
Writer: 5873.57

Testing dataset - average number of tokens in each class:
Actor: 4405.00
Animal: 1343.64
Artist: 4592.73
Book: 5188.36
Disease: 8027.67
Film: 4426.11
Food: 3592.38
Politician: 6032.93
Software: 4869.59
Writer: 6033.72


> 4) For each sentence in the document, remove punctuations and other special characters so that each sentence only contains English words and numbers. To make your life easier, you can make all words as lower cases. For each class, print out the first article's name and the processed first 40 words. (for both train and test dataset)

In [34]:
import re

TOKEN_PATTERN = re.compile(r"[a-z0-9]+")


def preprocess_sentence(sentence_text: str):
    return TOKEN_PATTERN.findall(sentence_text.lower())


def preprocess_document(text: str, nlp):
    processed_sentences = []
    doc = nlp(text)

    for sent in doc.sents:
        cleaned_tokens = preprocess_sentence(sent.text)
        if cleaned_tokens:
            processed_sentences.append(cleaned_tokens)

    return processed_sentences


def attach_processed_text(records, nlp):
    for record in records:
        processed_sentences = preprocess_document(record["text"], nlp)
        processed_tokens = [token for sentence in processed_sentences for token in sentence]

        record["processed_sentences"] = processed_sentences
        record["processed_tokens"] = processed_tokens
        record["processed_text"] = " ".join(processed_tokens)


def get_first_record_of_each_class(records):
    first_records = {}
    for record in records:
        label = record["label"]
        if label not in first_records:
            first_records[label] = record
    return first_records


attach_processed_text(train_records, nlp)
attach_processed_text(test_records, nlp)

train_first_records = get_first_record_of_each_class(train_records)
test_first_records = get_first_record_of_each_class(test_records)

print("Training dataset - first article in each class and its first 40 processed words:")
for label in sorted(train_first_records):
    record = train_first_records[label]
    preview_words = " ".join(record["processed_tokens"][:40])
    print(f"{label} | title: {record['title']}")
    print(f"first 40 words: {preview_words}")
    print()

print("Testing dataset - first article in each class and its first 40 processed words:")
for label in sorted(test_first_records):
    record = test_first_records[label]
    preview_words = " ".join(record["processed_tokens"][:40])
    print(f"{label} | title: {record['title']}")
    print(f"first 40 words: {preview_words}")
    print()


Training dataset - first article in each class and its first 40 processed words:
Actor | title: Laura_Bonarrigo
first 40 words: laura bonarrigo koffman born laura bonarrigo october 29 1964 is an american actress early life laura bonarrigo was born in brookline massachusetts she became a member of the shoestring players a professional children s theater group while still in grade

Animal | title: Lunaspis
first 40 words: lunaspis is an extinct genus of armor plated petalichthyid placoderm fish that lived in shallow marine environments of the early devonian period from approximately 409 1 to 402 5 million year ago fossils have been found in germany china and

Artist | title: Dan_Graham
first 40 words: daniel graham born march 31 1942 is an american artist writer and curator graham grew up in new jersey in 1964 he began directing the john daniels gallery in new york where he put on sol lewitt s first one

Book | title: Middlesex_(novel)
first 40 words: middlesex is a pulitzer prize winnin

### Task2 - Build language models

Before you go, you should do necessary preprocessing for training and testing text. For example, you should do sentence tokenization, removing special characters, replacing less frequency words as UNK (for example, you can try to use a cutoff of 10), making all words as lower characters. Fix your vocabulary size so that is not too large.

> 1) Based on the training dataset (collect all sentences in training dataset), build unigram, bigram, and trigram language models using Additive smoothing technique. It is encouraged to implement models by yourself.


In [35]:
import math
import random

if "processed_sentences" not in train_records[0]:
    attach_processed_text(train_records, nlp)
if "processed_sentences" not in test_records[0]:
    attach_processed_text(test_records, nlp)


def collect_all_sentences(records):
    all_sentences = []
    for record in records:
        all_sentences.extend(record["processed_sentences"])
    return all_sentences


train_sentences_raw = collect_all_sentences(train_records)
test_sentences_raw = collect_all_sentences(test_records)

UNK_TOKEN = "<UNK>"
START_TOKEN = "<s>"
END_TOKEN = "</s>"
MIN_FREQUENCY = 10
ADDITIVE_ALPHA = 1


def build_vocabulary(sentences, min_frequency: int = 10):
    token_counter = Counter()
    for sentence in sentences:
        token_counter.update(sentence)

    vocabulary = {
        token
        for token, count in token_counter.items()
        if count >= min_frequency
    }
    vocabulary.update({UNK_TOKEN, START_TOKEN, END_TOKEN})
    return vocabulary, token_counter


def replace_rare_and_oov_tokens(sentences, vocabulary):
    normalized_sentences = []
    for sentence in sentences:
        normalized_sentences.append([
            token if token in vocabulary else UNK_TOKEN
            for token in sentence
        ])
    return normalized_sentences


vocabulary, train_token_counter = build_vocabulary(train_sentences_raw, min_frequency=MIN_FREQUENCY)
train_sentences = replace_rare_and_oov_tokens(train_sentences_raw, vocabulary)
test_sentences = replace_rare_and_oov_tokens(test_sentences_raw, vocabulary)
vocabulary_list = sorted(vocabulary)
vocabulary_size = len(vocabulary_list)


class AdditiveNGramLanguageModel:
    def __init__(self, n: int, vocabulary, alpha: float = 1.0):
        self.n = n
        self.alpha = alpha
        self.vocabulary = set(vocabulary)
        self.vocabulary_list = sorted(vocabulary)
        self.vocabulary_size = len(self.vocabulary_list)
        self.ngram_counts = Counter()
        self.context_counts = Counter()

    def pad_sentence(self, sentence):
        if self.n == 1:
            return sentence + [END_TOKEN]
        start_tokens = [START_TOKEN] * (self.n - 1)
        return start_tokens + sentence + [END_TOKEN]

    def fit(self, sentences):
        for sentence in sentences:
            padded_sentence = self.pad_sentence(sentence)
            for i in range(len(padded_sentence) - self.n + 1):
                ngram = tuple(padded_sentence[i : i + self.n])
                context = ngram[:-1]
                self.ngram_counts[ngram] += 1
                self.context_counts[context] += 1

    def get_probability(self, token, context=None):
        if context is None:
            context = ()
        context = tuple(context[-(self.n - 1) :]) if self.n > 1 else ()
        if token not in self.vocabulary:
            token = UNK_TOKEN

        ngram = context + (token,)
        numerator = self.ngram_counts[ngram] + self.alpha
        denominator = self.context_counts[context] + self.alpha * self.vocabulary_size
        return numerator / denominator

    def sentence_log_probability(self, sentence):
        padded_sentence = self.pad_sentence(sentence)
        log_probability = 0.0
        for i in range(len(padded_sentence) - self.n + 1):
            ngram = tuple(padded_sentence[i : i + self.n])
            log_probability += math.log(self.get_probability(ngram[-1], ngram[:-1]))
        return log_probability

    def perplexity(self, sentences):
        total_log_probability = 0.0
        total_predicted_tokens = 0
        for sentence in sentences:
            padded_sentence = self.pad_sentence(sentence)
            total_log_probability += self.sentence_log_probability(sentence)
            total_predicted_tokens += len(padded_sentence) - self.n + 1
        return math.exp(-total_log_probability / total_predicted_tokens)

    def generate_sentence(self, max_length: int = 20, random_seed=None):
        if random_seed is not None:
            random.seed(random_seed)

        generated_tokens = []
        context = [START_TOKEN] * (self.n - 1) if self.n > 1 else []
        candidate_tokens = [token for token in self.vocabulary_list if token != START_TOKEN]

        for _ in range(max_length):
            probabilities = [self.get_probability(token, context) for token in candidate_tokens]
            next_token = random.choices(candidate_tokens, weights=probabilities, k=1)[0]
            if next_token == END_TOKEN:
                break
            generated_tokens.append(next_token)
            if self.n > 1:
                context = (context + [next_token])[-(self.n - 1) :]

        return generated_tokens


unigram_model = AdditiveNGramLanguageModel(n=1, vocabulary=vocabulary_list, alpha=ADDITIVE_ALPHA)
bigram_model = AdditiveNGramLanguageModel(n=2, vocabulary=vocabulary_list, alpha=ADDITIVE_ALPHA)
trigram_model = AdditiveNGramLanguageModel(n=3, vocabulary=vocabulary_list, alpha=ADDITIVE_ALPHA)

unigram_model.fit(train_sentences)
bigram_model.fit(train_sentences)
trigram_model.fit(train_sentences)

language_models = {
    "unigram": unigram_model,
    "bigram": bigram_model,
    "trigram": trigram_model,
}

print("Language model preprocessing and training are ready.")
print(f"Number of training sentences: {len(train_sentences)}")
print(f"Number of testing sentences: {len(test_sentences)}")
print(f"Raw training vocabulary size: {len(train_token_counter)}")
print(f"Final vocabulary size after cutoff={MIN_FREQUENCY}: {vocabulary_size}")
print(f"Additive smoothing alpha: {ADDITIVE_ALPHA}")


Language model preprocessing and training are ready.
Number of training sentences: 1831229
Number of testing sentences: 204705
Raw training vocabulary size: 320197
Final vocabulary size after cutoff=10: 75535
Additive smoothing alpha: 1


> 2) Report the perplexity of these 3 trained models on the testing dataset (again collect all sentences in training dataset) and explain your findings. 

In [36]:
perplexity_results = {}

for model_name, model in language_models.items():
    perplexity_results[model_name] = model.perplexity(test_sentences)

print("Perplexity on the testing dataset:")
for model_name, perplexity_value in perplexity_results.items():
    print(f"{model_name}: {perplexity_value:.4f}")

sorted_results = sorted(perplexity_results.items(), key=lambda item: item[1])
best_model_name, best_perplexity = sorted_results[0]
worst_model_name, worst_perplexity = sorted_results[-1]



Perplexity on the testing dataset:
unigram: 1466.5092
bigram: 1524.7748
trigram: 11275.1770


In [37]:
print("\nTrue Explanation:")
print(f"The best model is {best_model_name} with perplexity {best_perplexity:.4f}.")
print(f"The weakest model is {worst_model_name} with perplexity {worst_perplexity:.4f}.")
print("Lower perplexity means the model predicts the testing sentences better.")
print("Unigram usually performs worst because it ignores context, while bigram and trigram use local context.")
print("But now the results are uncommon,I think the reason is often data sparsity and the ALPHA is too large.")



True Explanation:
The best model is unigram with perplexity 1466.5092.
The weakest model is trigram with perplexity 11275.1770.
Lower perplexity means the model predicts the testing sentences better.
Unigram usually performs worst because it ignores context, while bigram and trigram use local context.
But now the results are uncommon,I think the reason is often data sparsity and the ALPHA is too large.


> 3) Use each built model to generate five sentences and explain these generated patterns.


In [38]:
NUM_SENTENCES_TO_GENERATE = 5
MAX_GENERATED_LENGTH = 20


def tokens_to_sentence(tokens):
    if not tokens:
        return "[empty sentence]"
    return " ".join(tokens)


generated_sentences = {}

for model_index, (model_name, model) in enumerate(language_models.items(), start=1):
    model_sentences = []
    for sentence_index in range(NUM_SENTENCES_TO_GENERATE):
        random_seed = model_index * 100 + sentence_index + 1
        generated_tokens = model.generate_sentence(max_length=MAX_GENERATED_LENGTH, random_seed=random_seed)
        model_sentences.append(tokens_to_sentence(generated_tokens))
    generated_sentences[model_name] = model_sentences

print("Generated sentences from each language model:\n")
for model_name, sentence_list in generated_sentences.items():
    print(f"{model_name} model:")
    for sentence_number, sentence_text in enumerate(sentence_list, start=1):
        print(f"{sentence_number}. {sentence_text}")
    print()




Generated sentences from each language model:

unigram model:
1. northern back when umberto in play blamed brings death raymond bibliography while <UNK> awakened a sport foreign inadvertently entering or
2. and of are rich of getting of much
3. with in running says blocked <UNK> of alma general to their albany a broadsheet the 9 burnt act among lessig
4. well their salam but eventually rio that 2016 appropriate to aired peintres evacuated over the not great emojis office on
5. they film taken was during specifically explanation antivirus film she and petered finds are his goal in this turkey but

bigram model:
1. all goes 150th dumped rentrak pondering jassi repin innate stuttering narco brockley meadville journalists louisiana copyright ponte cloves enables deceives
2. takeshi oregon height communicable pesci waterston human sower addams literatorul concetta kira nationalism sterilization lieutenancy suggestion faster bolshevist italie psoriatic
3. application specialise muir copperfi

In [39]:
print("True Pattern analysis:")
print("1. Unigram usually produces the loosest sentences because it ignores word order.")
print("2. Bigram usually gives more reasonable local phrases because it uses one-word context.")
print("3. Trigram perfomers badly due to data sparsity and the ALPHA being too large.")
print("4. N-gram models may still repeat words or lose global coherence.")

True Pattern analysis:
1. Unigram usually produces the loosest sentences because it ignores word order.
2. Bigram usually gives more reasonable local phrases because it uses one-word context.
3. Trigram perfomers badly due to data sparsity and the ALPHA being too large.
4. N-gram models may still repeat words or lose global coherence.


> 4) Train bigram and trigram model using kenlm and report the perplexities of these two. Compare results of your model and results from kenlm

In [40]:
import shutil
import subprocess
from pathlib import Path

try:
    import kenlm
except ImportError:
    kenlm = None


def ensure_command_exists(command_name: str):
    command_path = shutil.which(command_name)
    if command_path is None:
        raise FileNotFoundError(
            f"Command '{command_name}' was not found. Please install kenlm binaries and make sure this command is in PATH."
        )
    return command_path


def write_sentences_for_kenlm(sentences, output_path: Path):
    with output_path.open("w", encoding="utf-8") as f:
        for sentence in sentences:
            if sentence:
                f.write(" ".join(sentence) + "\n")


def build_kenlm_model(lmplz_path, build_binary_path, order: int, corpus_path: Path, arpa_path: Path, binary_path: Path):
    subprocess.run([
        lmplz_path,
        "-o", str(order),
        "--text", str(corpus_path),
        "--arpa", str(arpa_path),
        "--discount_fallback",
    ], check=True)

    subprocess.run([
        build_binary_path,
        str(arpa_path),
        str(binary_path),
    ], check=True)


def kenlm_corpus_perplexity(model, sentences):
    total_log10_probability = 0.0
    total_predicted_tokens = 0

    for sentence in sentences:
        sentence_text = " ".join(sentence)
        total_log10_probability += model.score(sentence_text, bos=True, eos=True)
        total_predicted_tokens += len(sentence) + 1

    return 10 ** (-total_log10_probability / total_predicted_tokens)


if kenlm is None:
    raise ImportError("The kenlm Python package is required. Please install kenlm before running this cell.")

lmplz_path = ensure_command_exists("lmplz")
build_binary_path = ensure_command_exists("build_binary")

KENLM_DIR = Path("kenlm_artifacts")
KENLM_DIR.mkdir(exist_ok=True)

train_corpus_path = KENLM_DIR / "train_corpus.txt"
bigram_arpa_path = KENLM_DIR / "bigram.arpa"
trigram_arpa_path = KENLM_DIR / "trigram.arpa"
bigram_binary_path = KENLM_DIR / "bigram.binary"
trigram_binary_path = KENLM_DIR / "trigram.binary"

write_sentences_for_kenlm(train_sentences, train_corpus_path)

build_kenlm_model(
    lmplz_path,
    build_binary_path,
    order=2,
    corpus_path=train_corpus_path,
    arpa_path=bigram_arpa_path,
    binary_path=bigram_binary_path,
)

build_kenlm_model(
    lmplz_path,
    build_binary_path,
    order=3,
    corpus_path=train_corpus_path,
    arpa_path=trigram_arpa_path,
    binary_path=trigram_binary_path,
)

kenlm_bigram_model = kenlm.Model(str(bigram_binary_path))
kenlm_trigram_model = kenlm.Model(str(trigram_binary_path))

kenlm_bigram_perplexity = kenlm_corpus_perplexity(kenlm_bigram_model, test_sentences)
kenlm_trigram_perplexity = kenlm_corpus_perplexity(kenlm_trigram_model, test_sentences)

if "perplexity_results" not in globals():
    perplexity_results = {
        model_name: model.perplexity(test_sentences)
        for model_name, model in language_models.items()
    }

our_bigram_perplexity = perplexity_results["bigram"]
our_trigram_perplexity = perplexity_results["trigram"]




python(48567) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
=== 1/5 Counting and sorting n-grams ===
Reading fd 3
----5---10---15---20---25---30---35---40---45---50---55---60---65---70---75---80---85---90---95--100
****************************************************************************************************
Unigram tokens 40684581 types 75536
=== 2/5 Calculating and sorting adjusted counts ===
Chain sizes: 1:906432 2:13742989312
Statistics:
1 75536 D1=0.331617 D2=0.874885 D3+=1.16764
2 6460653 D1=0.716792 D2=1.0787 D3+=1.38539
Memory estimate for binary LM:
type     MB
probing 112 assuming -p 1.5
probing 113 assuming -r models -p 1.5
trie     38 without quantization
trie     20 assuming -q 8 -b 8 quantization 
trie     38 assuming -a 22 array pointer compression
trie     20 assuming -a 22 -q 8 -b 8 array pointer compression and quantization
=== 3/5 Calculating and sorting initial probabilities ===
Chain sizes: 1:906432 2:103370448
=== 4/5 Ca

In [41]:
print("True KenLM perplexity on the testing dataset:")
print(f"kenlm bigram: {kenlm_bigram_perplexity:.4f}")
print(f"kenlm trigram: {kenlm_trigram_perplexity:.4f}")

print("\nComparison with my additive-smoothing implementation:")
print(f"my bigram:    {our_bigram_perplexity:.4f}")
print(f"kenlm bigram: {kenlm_bigram_perplexity:.4f}")
print(f"my trigram:   {our_trigram_perplexity:.4f}")
print(f"kenlm trigram:{kenlm_trigram_perplexity:.4f}")

print("\nComparison analysis:")
print("This part now trains the language models with the official KenLM toolchain, so the comparison is methodologically aligned with the assignment requirement.")
print("KenLM obtains lower perplexity,meaning its estimation and backoff handling are stronger than the simple hand-written implementation.")

True KenLM perplexity on the testing dataset:
kenlm bigram: 353.4368
kenlm trigram: 248.4485

Comparison with my additive-smoothing implementation:
my bigram:    1524.7748
kenlm bigram: 353.4368
my trigram:   11275.1770
kenlm trigram:248.4485

Comparison analysis:
This part now trains the language models with the official KenLM toolchain, so the comparison is methodologically aligned with the assignment requirement.
KenLM obtains lower perplexity,meaning its estimation and backoff handling are stronger than the simple hand-written implementation.


## Task3 - Build NB/LR classifiers

> 1) Build a Naive Bayes classifier (with Laplace smoothing) and test your model on test dataset

In [42]:
from collections import defaultdict

if "processed_tokens" not in train_records[0]:
    attach_processed_text(train_records, nlp)
if "processed_tokens" not in test_records[0]:
    attach_processed_text(test_records, nlp)


def normalize_document_tokens(tokens, vocabulary):
    return [token if token in vocabulary else UNK_TOKEN for token in tokens]


train_documents = [normalize_document_tokens(record["processed_tokens"], vocabulary) for record in train_records]
train_labels = [record["label"] for record in train_records]
test_documents = [normalize_document_tokens(record["processed_tokens"], vocabulary) for record in test_records]
test_labels = [record["label"] for record in test_records]


class MultinomialNaiveBayes:
    def __init__(self, alpha: float = 1.0):
        self.alpha = alpha
        self.classes_ = []
        self.vocabulary_ = set()
        self.class_document_counts = Counter()
        self.class_token_counts = Counter()
        self.token_counts_per_class = defaultdict(Counter)
        self.log_class_priors = {}
        self.log_token_likelihoods = defaultdict(dict)
        self.default_log_token_likelihood = {}

    def fit(self, documents, labels, vocabulary):
        self.vocabulary_ = set(vocabulary)
        self.classes_ = sorted(set(labels))
        vocabulary_size = len(self.vocabulary_)
        total_documents = len(documents)

        for document_tokens, label in zip(documents, labels):
            self.class_document_counts[label] += 1
            document_counter = Counter(document_tokens)
            for token, count in document_counter.items():
                self.token_counts_per_class[label][token] += count
                self.class_token_counts[label] += count

        for label in self.classes_:
            self.log_class_priors[label] = math.log(self.class_document_counts[label] / total_documents)

        for label in self.classes_:
            denominator = self.class_token_counts[label] + self.alpha * vocabulary_size
            self.default_log_token_likelihood[label] = math.log(self.alpha / denominator)
            for token in self.vocabulary_:
                probability = (self.token_counts_per_class[label][token] + self.alpha) / denominator
                self.log_token_likelihoods[label][token] = math.log(probability)

    def predict_one(self, document_tokens):
        document_counter = Counter(document_tokens)
        class_scores = {}

        for label in self.classes_:
            score = self.log_class_priors[label]
            for token, count in document_counter.items():
                token_log_probability = self.log_token_likelihoods[label].get(token, self.default_log_token_likelihood[label])
                score += count * token_log_probability
            class_scores[label] = score

        return max(class_scores, key=class_scores.get)

    def predict(self, documents):
        return [self.predict_one(document_tokens) for document_tokens in documents]


nb_classifier = MultinomialNaiveBayes(alpha=1.0)
nb_classifier.fit(train_documents, train_labels, vocabulary_list)
nb_predictions = nb_classifier.predict(test_documents)
nb_accuracy = sum(pred == gold for pred, gold in zip(nb_predictions, test_labels)) / len(test_labels)

print("Naive Bayes classifier has been trained and tested on the test dataset.")
print(f"Test accuracy: {nb_accuracy:.4f}")
print("\nFirst 10 prediction examples:")
for index, (predicted_label, gold_label, record) in enumerate(zip(nb_predictions, test_labels, test_records), start=1):
    if index > 10:
        break
    print(f"{index}. title={record['title']} | predicted={predicted_label} | gold={gold_label}")


Naive Bayes classifier has been trained and tested on the test dataset.
Test accuracy: 0.9380

First 10 prediction examples:
1. title=Spessard_Holland | predicted=Politician | gold=Politician
2. title=Jørgen-Frantz_Jacobsen | predicted=Book | gold=Writer
3. title=See_You_Tomorrow_(novel) | predicted=Book | gold=Book
4. title=The_Muppets_(film) | predicted=Film | gold=Film
5. title=Four_Weddings_and_a_Funeral | predicted=Film | gold=Film
6. title=Kenilworth_(novel) | predicted=Book | gold=Book
7. title=The_Lord_of_the_Rings | predicted=Writer | gold=Book
8. title=Romulo_Espaldon | predicted=Politician | gold=Politician
9. title=Jose_Diokno | predicted=Politician | gold=Politician
10. title=Rafail_Levitsky | predicted=Artist | gold=Artist


> 2) Build a LR classifier. This question seems to be challenging. We did not directly provide features for samples. But just use your own method to build useful features. You may need to split the training dataset into train and validation so that some involved parameters can be tuned. 

In [43]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score
from sklearn.model_selection import train_test_split

if "processed_text" not in train_records[0]:
    attach_processed_text(train_records, nlp)
if "processed_text" not in test_records[0]:
    attach_processed_text(test_records, nlp)

train_texts = [record["processed_text"] for record in train_records]
train_labels = [record["label"] for record in train_records]
test_texts = [record["processed_text"] for record in test_records]
test_labels = [record["label"] for record in test_records]

lr_train_texts, lr_valid_texts, lr_train_labels, lr_valid_labels = train_test_split(
    train_texts,
    train_labels,
    test_size=0.2,
    random_state=42,
    stratify=train_labels,
)

lr_param_grid = [
    {"ngram_range": (1, 1), "min_df": 3, "C": 1.0},
    {"ngram_range": (1, 2), "min_df": 3, "C": 1.0},
    {"ngram_range": (1, 2), "min_df": 5, "C": 1.0},
    {"ngram_range": (1, 2), "min_df": 3, "C": 2.0},
]

lr_validation_results = []
best_lr_score = -1.0
best_lr_config = None

for config in lr_param_grid:
    vectorizer = TfidfVectorizer(
        lowercase=False,
        tokenizer=str.split,
        preprocessor=None,
        token_pattern=None,
        ngram_range=config["ngram_range"],
        min_df=config["min_df"],
    )

    X_train = vectorizer.fit_transform(lr_train_texts)
    X_valid = vectorizer.transform(lr_valid_texts)

    lr_model = LogisticRegression(C=config["C"], max_iter=500, solver="lbfgs")
    lr_model.fit(X_train, lr_train_labels)

    valid_predictions = lr_model.predict(X_valid)
    valid_accuracy = accuracy_score(lr_valid_labels, valid_predictions)
    valid_macro_f1 = f1_score(lr_valid_labels, valid_predictions, average="macro")

    lr_validation_results.append({
        "config": config,
        "valid_accuracy": valid_accuracy,
        "valid_macro_f1": valid_macro_f1,
    })

    if valid_macro_f1 > best_lr_score:
        best_lr_score = valid_macro_f1
        best_lr_config = config

final_lr_vectorizer = TfidfVectorizer(
    lowercase=False,
    tokenizer=str.split,
    preprocessor=None,
    token_pattern=None,
    ngram_range=best_lr_config["ngram_range"],
    min_df=best_lr_config["min_df"],
)

X_train_full = final_lr_vectorizer.fit_transform(train_texts)
X_test = final_lr_vectorizer.transform(test_texts)

lr_classifier = LogisticRegression(C=best_lr_config["C"], max_iter=500, solver="lbfgs")
lr_classifier.fit(X_train_full, train_labels)
lr_predictions = lr_classifier.predict(X_test)
lr_test_accuracy = accuracy_score(test_labels, lr_predictions)




In [44]:
print("Validation results for different LR settings:")
for result_row in lr_validation_results:
    print(
        f"config={result_row['config']} | "
        f"valid_accuracy={result_row['valid_accuracy']:.4f} | "
        f"valid_macro_f1={result_row['valid_macro_f1']:.4f}"
    )

print("\nBest LR configuration:")
print(best_lr_config)
print(f"Best validation macro-F1: {best_lr_score:.4f}")
print("\nFinal Logistic Regression test result:")
print(f"Test accuracy: {lr_test_accuracy:.4f}")
print("\nFirst 10 LR prediction examples:")
for index, (predicted_label, gold_label, record) in enumerate(zip(lr_predictions, test_labels, test_records), start=1):
    if index > 10:
        break
    print(f"{index}. title={record['title']} | predicted={predicted_label} | gold={gold_label}")

Validation results for different LR settings:
config={'ngram_range': (1, 1), 'min_df': 3, 'C': 1.0} | valid_accuracy=0.9378 | valid_macro_f1=0.8099
config={'ngram_range': (1, 2), 'min_df': 3, 'C': 1.0} | valid_accuracy=0.9400 | valid_macro_f1=0.8061
config={'ngram_range': (1, 2), 'min_df': 5, 'C': 1.0} | valid_accuracy=0.9428 | valid_macro_f1=0.8164
config={'ngram_range': (1, 2), 'min_df': 3, 'C': 2.0} | valid_accuracy=0.9567 | valid_macro_f1=0.8730

Best LR configuration:
{'ngram_range': (1, 2), 'min_df': 3, 'C': 2.0}
Best validation macro-F1: 0.8730

Final Logistic Regression test result:
Test accuracy: 0.9560

First 10 LR prediction examples:
1. title=Spessard_Holland | predicted=Politician | gold=Politician
2. title=Jørgen-Frantz_Jacobsen | predicted=Book | gold=Writer
3. title=See_You_Tomorrow_(novel) | predicted=Book | gold=Book
4. title=The_Muppets_(film) | predicted=Film | gold=Film
5. title=Four_Weddings_and_a_Funeral | predicted=Film | gold=Film
6. title=Kenilworth_(novel) | 

> 3) Report Micro-F1 score and Macro-F1 score for these classifiers on testing dataset explain your results.

In [45]:
from sklearn.metrics import classification_report

if "nb_predictions" not in globals():
    raise RuntimeError("Please run the Naive Bayes cell first so that nb_predictions is available.")
if "lr_predictions" not in globals():
    raise RuntimeError("Please run the Logistic Regression cell first so that lr_predictions is available.")

nb_micro_f1 = f1_score(test_labels, nb_predictions, average="micro")
nb_macro_f1 = f1_score(test_labels, nb_predictions, average="macro")
lr_micro_f1 = f1_score(test_labels, lr_predictions, average="micro")
lr_macro_f1 = f1_score(test_labels, lr_predictions, average="macro")



In [46]:
print("F1-score results on the testing dataset:")
print(f"Naive Bayes    | Micro-F1: {nb_micro_f1:.4f} | Macro-F1: {nb_macro_f1:.4f}")
print(f"Logistic Reg.  | Micro-F1: {lr_micro_f1:.4f} | Macro-F1: {lr_macro_f1:.4f}")

print("\nNaive Bayes classification report:")
print(classification_report(test_labels, nb_predictions, digits=4))
print("Logistic Regression classification report:")
print(classification_report(test_labels, lr_predictions, digits=4))

print("Explanation:")
better_micro_model = "Naive Bayes" if nb_micro_f1 >= lr_micro_f1 else "Logistic Regression"
better_macro_model = "Naive Bayes" if nb_macro_f1 >= lr_macro_f1 else "Logistic Regression"
print(f"For Micro-F1, the better model is {better_micro_model}.")
print(f"For Macro-F1, the better model is {better_macro_model}.")
print("Micro-F1 is much higher than Macro-F1, meaning that the model is usually better on large classes than on small classes.")
print("This is common here because the dataset is imbalanced.")


F1-score results on the testing dataset:
Naive Bayes    | Micro-F1: 0.9380 | Macro-F1: 0.8331
Logistic Reg.  | Micro-F1: 0.9560 | Macro-F1: 0.9405

Naive Bayes classification report:
              precision    recall  f1-score   support

       Actor     0.0000    0.0000    0.0000         1
      Animal     1.0000    0.9091    0.9524        11
      Artist     0.9831    0.9206    0.9508        63
        Book     0.8673    0.8376    0.8522       117
     Disease     0.8500    0.9444    0.8947        18
        Film     0.9696    0.9696    0.9696       296
        Food     1.0000    1.0000    1.0000        16
  Politician     0.9710    0.9608    0.9659       383
    Software     0.9310    1.0000    0.9643        27
      Writer     0.7308    0.8382    0.7808        68

    accuracy                         0.9380      1000
   macro avg     0.8303    0.8380    0.8331      1000
weighted avg     0.9394    0.9380    0.9383      1000

Logistic Regression classification report:
              p

/Users/hurior/.local/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/Users/hurior/.local/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/Users/hurior/.local/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


### Task 4 - Classify documents using embeddings

> For each document (both training and testing documents), you have several choices to generate a document embedding from the embedding we trained in Task 1 (**Just choose one of them**):

> - Use the average of known static embeddings of all words in each document. Or use the first paragraph’s words and take an average on these embeddings.
> - Use the doc2vec algorithm to present each document.
> - Use modern embedding like Qwen-embedding 0.6b ...

> Build a classifer on training documents and testing this classifer on testing documents. Since you have the ground truth, you can use the Micro/Macro F1-score to measure the performance of these choices on testing documents.

In [47]:
import numpy as np

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, f1_score
from sklearn.model_selection import train_test_split

try:
    from gensim.models import Word2Vec
except ImportError as exc:
    raise ImportError("gensim is required for Task4. Please install gensim before running this cell.") from exc

if "processed_tokens" not in train_records[0]:
    attach_processed_text(train_records, nlp)
if "processed_tokens" not in test_records[0]:
    attach_processed_text(test_records, nlp)

task4_train_documents = [record["processed_tokens"] for record in train_records]
task4_train_labels = [record["label"] for record in train_records]
task4_test_documents = [record["processed_tokens"] for record in test_records]
task4_test_labels = [record["label"] for record in test_records]

TASK4_VECTOR_SIZE = 50
TASK4_WINDOW = 5
TASK4_MIN_COUNT = 10
TASK4_WORKERS = 4
TASK4_SEED = 42

task4_word2vec = Word2Vec(
    sentences=task4_train_documents,
    vector_size=TASK4_VECTOR_SIZE,
    window=TASK4_WINDOW,
    min_count=TASK4_MIN_COUNT,
    workers=TASK4_WORKERS,
    sg=1,
    seed=TASK4_SEED,
)


def document_to_average_embedding(document_tokens, word2vec_model):
    known_vectors = []
    for token in document_tokens:
        if token in word2vec_model.wv:
            known_vectors.append(word2vec_model.wv[token])
    if not known_vectors:
        return np.zeros(word2vec_model.vector_size, dtype=np.float32)
    return np.mean(known_vectors, axis=0)


def build_document_embedding_matrix(documents, word2vec_model):
    return np.vstack([
        document_to_average_embedding(document_tokens, word2vec_model)
        for document_tokens in documents
    ])


task4_train_embeddings = build_document_embedding_matrix(task4_train_documents, task4_word2vec)
task4_test_embeddings = build_document_embedding_matrix(task4_test_documents, task4_word2vec)

task4_X_train, task4_X_valid, task4_y_train, task4_y_valid = train_test_split(
    task4_train_embeddings,
    task4_train_labels,
    test_size=0.2,
    random_state=42,
    stratify=task4_train_labels,
)

task4_c_candidates = [0.5, 1.0, 2.0, 5.0]
task4_validation_results = []
task4_best_c = None
task4_best_macro_f1 = -1.0

for c_value in task4_c_candidates:
    candidate_model = LogisticRegression(C=c_value, max_iter=500, solver="lbfgs")
    candidate_model.fit(task4_X_train, task4_y_train)

    valid_predictions = candidate_model.predict(task4_X_valid)
    valid_micro_f1 = f1_score(task4_y_valid, valid_predictions, average="micro")
    valid_macro_f1 = f1_score(task4_y_valid, valid_predictions, average="macro")

    task4_validation_results.append({
        "C": c_value,
        "valid_micro_f1": valid_micro_f1,
        "valid_macro_f1": valid_macro_f1,
    })

    if valid_macro_f1 > task4_best_macro_f1:
        task4_best_macro_f1 = valid_macro_f1
        task4_best_c = c_value

task4_classifier = LogisticRegression(C=task4_best_c, max_iter=500, solver="lbfgs")
task4_classifier.fit(task4_train_embeddings, task4_train_labels)
task4_predictions = task4_classifier.predict(task4_test_embeddings)

task4_micro_f1 = f1_score(task4_test_labels, task4_predictions, average="micro")
task4_macro_f1 = f1_score(task4_test_labels, task4_predictions, average="macro")

print("Validation results for embedding-based Logistic Regression:")
for result_row in task4_validation_results:
    print(
        f"C={result_row['C']} | "
        f"valid_micro_f1={result_row['valid_micro_f1']:.4f} | "
        f"valid_macro_f1={result_row['valid_macro_f1']:.4f}"
    )




Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_fl

Validation results for embedding-based Logistic Regression:
C=0.5 | valid_micro_f1=0.9356 | valid_macro_f1=0.8091
C=1.0 | valid_micro_f1=0.9428 | valid_macro_f1=0.8365
C=2.0 | valid_micro_f1=0.9511 | valid_macro_f1=0.8734
C=5.0 | valid_micro_f1=0.9594 | valid_macro_f1=0.9035


In [48]:
print("\nBest C selected from validation:")
print(task4_best_c)
print("\nEmbedding-based classification results on the testing dataset:")
print(f"Micro-F1: {task4_micro_f1:.4f}")
print(f"Macro-F1: {task4_macro_f1:.4f}")
print("\nClassification report:")
print(classification_report(task4_test_labels, task4_predictions, digits=4))
print("Explanation:")
print("Documents are represented by the average of their word embeddings.")
print("This captures some semantic similarity, but still ignores word order.")
print("Micro-F1 is higher than Macro-F1, meaning that the model is usually stronger on large classes than on small classes.")


Best C selected from validation:
5.0

Embedding-based classification results on the testing dataset:
Micro-F1: 0.9490
Macro-F1: 0.8430

Classification report:
              precision    recall  f1-score   support

       Actor     0.0000    0.0000    0.0000         1
      Animal     0.9167    1.0000    0.9565        11
      Artist     0.9818    0.8571    0.9153        63
        Book     0.8870    0.8718    0.8793       117
     Disease     1.0000    0.8889    0.9412        18
        Film     0.9732    0.9831    0.9782       296
        Food     1.0000    1.0000    1.0000        16
  Politician     0.9617    0.9843    0.9729       383
    Software     0.9630    0.9630    0.9630        27
      Writer     0.8235    0.8235    0.8235        68

    accuracy                         0.9490      1000
   macro avg     0.8507    0.8372    0.8430      1000
weighted avg     0.9481    0.9490    0.9482      1000

Explanation:
Documents are represented by the average of their word embeddings.
T

/Users/hurior/.local/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/Users/hurior/.local/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/Users/hurior/.local/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


#### upload your homework: use html file format
!jupyter nbconvert assignment-01-XXX-YYY.ipynb --to html --template classic --embed-images

In [49]:
!jupyter nbconvert assignment-01-24307140070-王志昊.ipynb --to html --template classic --embed-images

python(48747) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.


[NbConvertApp] Converting notebook assignment-01-24307140070-王志昊.ipynb to html
[NbConvertApp] Writing 423504 bytes to assignment-01-24307140070-王志昊.html
